In [0]:
#Essa seção do Notebook é focada na normalização da tabela bronze.tb_movies_info
from pyspark.sql import functions as F
from pyspark.sql.window import Window
df = spark.table("medallion.bronze.tb_movies_info") #Aqui estamos criando a variavel df que armazena a tabela bronze.tb_movies_info
df_movies = (
    df.withColumnRenamed("id","id_filme") #Traduzindo a coluna id para id_filme 
    .withColumnRenamed("title","titulo") #Traduzindo a coluna title para titulo
    .withColumnRenamed("original_title","titulo_original") #Traduzindo a coluna original_title para titulo_original
    .withColumnRenamed("release_date","data_lancamento") #Traduzindo a coluna release_date para data_lançamento
    .withColumnRenamed("runtime","duracao_minutos") #Traduzindo a coluna runtime para duracao_minutos
    .withColumnRenamed("original_language","idioma_original") #Traduzindo original_language para idioma_original
    .withColumnRenamed("status","status_filme") #Traduzindo status para status_filme
    .withColumnRenamed("overview","sinopse") #Traduzindo overview para sinopse
    .withColumnRenamed("tagline","frase_divulgacao") #Traduzindo tagline para frase_divulgacao
    .withColumn(
    "duracao_minutos",
    F.expr("try_cast(duracao_minutos as int)")
    )
    .withColumn(
        "status_filme",
        F.initcap(
            F.trim(
            F.regexp_replace(
            F.regexp_replace(F.col("status_filme"), r"[-_]+", " "),
            r"\s+", " "
        )
        )
        )
    )# Padronizo o texto para Capitalize, substituo hífens e underscores por espaços e removo espaços desnecessários.
    .withColumn(
        "status_filme",
        F.when(F.col("status_filme") == "Released", "Lançado")
        .when(F.col("status_filme") == "Post Production", "Pós-Produção")
        .when(F.col("status_filme") == "In Production", "Em Produção")
        .when(F.col("status_filme") == "Planned", "Planejado")
        .when(F.col("status_filme") == "Rumored", "Rumores")
        .when(F.col("status_filme") == "Canceled", "Cancelado")  
        .otherwise("Não Informado") #Quando não tiver a um dos casos informados colocar Não Informado
    ) #Seção focada em traduzir as celulas da tabela conforme solicitado no pdf
    .withColumn(
        "data_lancamento",
        F.coalesce(
            F.expr("try_to_date(data_lancamento, 'yyyy-MM-dd')"),
            F.expr("try_to_date(data_lancamento, 'dd/MM/yyyy')"),
            F.expr("try_to_date(data_lancamento, 'MM/dd/yyyy')"),
            F.expr("try_to_date(data_lancamento, 'yyyy/MM/dd')"),
            F.expr("try_to_date(data_lancamento, 'dd-MM-yyyy')"),
            F.expr("try_to_date(data_lancamento, 'MM-dd-yyyy')")
        )
    ) #Padronização de todas as maneiras comuns de representação de datas
    .withColumn(
        "ano_lancamento",
        F.year(F.col("data_lancamento"))
    ) #Coluna derivada: extrai o ano a partir de data_lancamento
  
)

window_dedup = Window.partitionBy("id_filme").orderBy(F.col("ingestion_datetime").desc(), F.col("_qualidade").desc()) #Janela responsavel por tirar duplicação e ordenar pela mais nova com criterio de desempate
df_movies = df_movies.withColumn(
    "_qualidade",
    F.when(F.col("data_lancamento").isNotNull(), 1).otherwise(0) +
    F.when(F.col("duracao_minutos").isNotNull(), 1).otherwise(0) +
    F.when(F.col("sinopse").isNotNull(), 1).otherwise(0) +
    F.when(F.col("frase_divulgacao").isNotNull(), 1).otherwise(0)
) #Coluna de apoio: pontua quantos campos relevantes estão preenchidos, usada como critério de desempate
df_movies = (
    df_movies
    .withColumn("row_number", F.row_number().over(window_dedup))
    .filter(F.col("row_number") == 1)
    .drop("row_number", "_qualidade")
) #Aqui aplicamos a janela pra tirar a duplicação

df_movies.write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable("medallion.silver.tb_info_filmes")#Criação Silver
print("Tabela silver.tb_info_filmes criada corretamente")


In [0]:
from pyspark.sql.window import Window

#Busca a cotação mais recente disponível na Bronze.
df_cotacao_atual = (
    spark.table("medallion.bronze.tb_cotacao_dolar")
    .withColumn(
        "_data_hora",
        F.expr("try_cast(dataHoraCotacao AS TIMESTAMP)")
    )
    .withColumn(
        "cotacaoCompra",
        F.expr("try_cast(cotacaoCompra AS DECIMAL(10,4))")
    )
    .filter(
        F.col("_data_hora").isNotNull()
        & (F.col("cotacaoCompra") > 0)
    )
    .orderBy(F.col("_data_hora").desc())
    .select("cotacaoCompra")
)

registro_cotacao = df_cotacao_atual.first()

if registro_cotacao is None:
    raise ValueError(
        "Não existe cotação válida na Bronze. "
        "Execute primeiro a ingestão da tb_cotacao_dolar."
    )

cotacao_dolar = registro_cotacao["cotacaoCompra"]

print("Cotação utilizada:", cotacao_dolar)

In [0]:
#Essa seção do Notebook é focada na normalização da tabela bronze.tb_movies_financials

from pyspark.sql import functions as F

df = spark.table("medallion.bronze.tb_movies_financials") #Aqui estamos criando a variavel df que armazena a tabela bronze.tb_movies_financials

#Mesma extração numérica e multiplicadores K/M/B da referência.
def higienizar(coluna):
    texto = F.upper(F.trim(F.col(coluna)))

    numero = F.expr(f"""
        try_cast(
            regexp_extract(
                regexp_replace(upper(trim({coluna})), ',', ''),
                '(-?[0-9]+(?:\\.[0-9]+)?)',
                1
            )
            as decimal(20,4)
        )
    """)

    valor = (
        F.when(texto.rlike(r"K\s*$"), numero * 1_000)
         .when(texto.rlike(r"M\s*$"), numero * 1_000_000)
         .when(texto.rlike(r"B\s*$"), numero * 1_000_000_000)
         .otherwise(numero)
    )

    return F.when(valor > 0, valor)



df_financials = (

    df.withColumnRenamed("id", "id_filme") #Traduzindo a coluna id para id_filme

    .withColumn("orcamento_usd", higienizar("budget")) #Higienizando e convertendo o orçamento para um valor numerico em dólar

    .withColumn("receita_usd", higienizar("revenue")) #Higienizando e convertendo a receita para um valor numerico em dólar

    #Deduplica por ingestão, qualidade e hash dos valores originais.
    .withColumn("_qualidade",
        F.when(F.col("orcamento_usd").isNotNull(), 1).otherwise(0)
        + F.when(F.col("receita_usd").isNotNull(), 1).otherwise(0))
    .withColumn("_rn", F.row_number().over(
        Window.partitionBy("id_filme").orderBy(
            F.col("ingestion_datetime").desc(), F.col("_qualidade").desc(),
            F.xxhash64("budget", "revenue").desc())))
    .filter(F.col("_rn") == 1)
    .drop("budget", "revenue", "_qualidade", "_rn")
    .withColumn("orcamento_usd", F.col("orcamento_usd").cast("decimal(18,2)"))
    .withColumn("receita_usd", F.col("receita_usd").cast("decimal(18,2)"))

    # Conversão de USD para BRL
    .withColumn(
        "orcamento_brl",
        (F.col("orcamento_usd") * F.lit(cotacao_dolar)).cast("decimal(18,2)")
    ) #Converte o orçamento de dólar para real utilizando a cotação obtida da Bronze

    .withColumn(
        "receita_brl",
        (F.col("receita_usd") * F.lit(cotacao_dolar)).cast("decimal(18,2)")
    ) #Converte a receita de dólar para real utilizando a cotação obtida da Bronze

    # Cálculo do lucro em USD
    .withColumn(
        "lucro_usd",
        (
            F.col("receita_usd")
            - F.col("orcamento_usd")
        ).cast("decimal(18,2)")
    ) #Calcula o lucro em dólar subtraindo o orçamento da receita; se uma parcela estiver ausente, o lucro fica NULL

    # Cálculo do lucro em BRL
    .withColumn(
        "lucro_brl",
        (
            F.col("receita_brl")
            - F.col("orcamento_brl")
        ).cast("decimal(18,2)")
    ) #Calcula o lucro em real subtraindo o orçamento da receita; se uma parcela estiver ausente, o lucro fica NULL

    #Mesma regra da referência: lucro / orçamento * 100.
    #O nome foi mantido para compatibilidade; esta razão mede retorno sobre orçamento.
    .withColumn(
        "margem_lucro_percentual",
        ((F.col("receita_usd") - F.col("orcamento_usd"))
         / F.col("orcamento_usd") * 100).cast("decimal(18,2)")
    )
)

df_financials = df_financials.select(
    [col for col in df_financials.columns if col != "ingestion_datetime"]
    + ["ingestion_datetime"]
) #Reorganiza as colunas mantendo ingestion_datetime como a ultima coluna

df_financials.write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable("medallion.silver.tb_financeiro_filmes") #Criação da tabela Silver sobrescrevendo os dados existentes

print("Tabela silver.tb_financeiro_filmes criada corretamente") #Mensagem informando que a tabela Silver foi criada/atualizada corretamente


In [0]:
#Essa seção do Notebook é focada na normalização da tabela bronze.tb_movies_metrics

from pyspark.sql import functions as F

df = spark.table("medallion.bronze.tb_movies_metrics") #Aqui estamos criando a variavel df que armazena a tabela bronze.tb_movies_metrics

df_metrics = (

    df.withColumnRenamed("id", "id_filme") #Traduzindo a coluna id para id_filme

      .withColumnRenamed("popularity", "popularidade") #Traduzindo a coluna popularity para popularidade

      .withColumnRenamed("vote_average", "nota_media_tmdb") #Traduzindo vote_average para nota_media_tmdb

      .withColumnRenamed("vote_count", "qtd_votos_tmdb") #Traduzindo vote_count para qtd_votos_tmdb

      .withColumnRenamed("averageRating", "nota_media_imdb") #Traduzindo averageRating para nota_media_imdb

      .withColumnRenamed("numVotes", "qtd_votos_imdb") #Traduzindo numVotes para qtd_votos_imdb
      
      #Mesma regra da referência: troca vírgula por ponto e aplica conversão segura.
      .withColumn("popularidade", F.expr("try_cast(regexp_replace(trim(popularidade), ',', '.') AS DOUBLE)"))

      .withColumn(
        "nota_media_tmdb",
        F.expr("try_cast(nota_media_tmdb AS DOUBLE)")
      )
      #Convertendo nota_media_tmdb para DOUBLE.

      .withColumn(
        "qtd_votos_tmdb",
        F.expr("try_cast(qtd_votos_tmdb AS INT)")
      )
      #Convertendo qtd_votos_tmdb para INT.

      .withColumn(
        "nota_media_imdb",
        F.expr("try_cast(nota_media_imdb AS DOUBLE)")
      )
      #Convertendo nota_media_imdb para DOUBLE.

      .withColumn(
        "qtd_votos_imdb",
        F.expr("try_cast(qtd_votos_imdb AS INT)")
      )
      #Convertendo qtd_votos_imdb para INT.

      .withColumn("popularidade", F.when(F.col("popularidade") >= 0, F.col("popularidade")))
      .withColumn("nota_media_tmdb", F.when(F.col("nota_media_tmdb").between(0, 10), F.col("nota_media_tmdb")))
      .withColumn("nota_media_imdb", F.when(F.col("nota_media_imdb").between(0, 10), F.col("nota_media_imdb")))
      .withColumn("qtd_votos_tmdb", F.when(F.col("qtd_votos_tmdb") >= 0, F.col("qtd_votos_tmdb")))
      .withColumn("qtd_votos_imdb", F.when(F.col("qtd_votos_imdb") >= 0, F.col("qtd_votos_imdb")))
)

#Mesma prioridade de deduplicação da referência.
df_metrics = df_metrics.withColumn("_qualidade",
    F.when(F.col("popularidade").isNotNull(), 1).otherwise(0)
    + F.when(F.col("nota_media_tmdb").isNotNull(), 1).otherwise(0)
    + F.when(F.col("qtd_votos_tmdb").isNotNull(), 1).otherwise(0)
    + F.when(F.col("nota_media_imdb").isNotNull(), 1).otherwise(0)
    + F.when(F.col("qtd_votos_imdb").isNotNull(), 1).otherwise(0))
window_dedup = Window.partitionBy("id_filme").orderBy(F.col("ingestion_datetime").desc(), F.col("_qualidade").desc())
df_metrics = (df_metrics.withColumn("_rn", F.row_number().over(window_dedup))
    .filter(F.col("_rn") == 1).drop("_rn", "_qualidade"))

df_metrics.write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable("medallion.silver.tb_metricas_engajamento")
#Criação da tabela Silver, sobrescrevendo os dados existentes e atualizando o schema, conforme as transformações realizadas no DataFrame.

print("Tabela silver.tb_metricas_engajamento criada corretamente")
#Mensagem informando que a tabela Silver foi criada/atualizada corretamente.


In [0]:
#Essa seção do Notebook é focada na normalização da tabela bronze.tb_movies_reviews

from pyspark.sql import functions as F

df = spark.table("medallion.bronze.tb_movies_reviews") #Aqui estamos criando a variavel df que armazena a tabela bronze.tb_movies_reviews

df_review = (

    df.dropDuplicates(["id", "nome", "nota", "comentario"])
      .withColumnRenamed("id", "id_filme") #Traduzindo a coluna id para id_filme

      .withColumnRenamed("nome", "nome_usuario") #Traduzindo a coluna nome para nome_usuario

      .withColumnRenamed("nota", "nota_usuario") #Traduzindo a coluna nota para nota_usuario

      .withColumnRenamed("comentario", "comentario_usuario") #Traduzindo a coluna comentario para comentario_usuario

      .withColumn(
          "comentario_usuario",
          F.when(
              F.col("comentario_usuario").isNull() |
              (F.trim(F.col("comentario_usuario")) == ""),
              "Sem comentário"
          )
          .otherwise(F.trim(F.col("comentario_usuario")))
      )
      #Substituindo comentarios nulos ou vazios pelo texto padronizado "Sem comentário"

      # CORREÇÃO: converter antes de comparar com os limites da escala.
      .withColumn("nota_usuario", F.expr("try_cast(nota_usuario AS DOUBLE)"))
      .withColumn(
          "nota_usuario",
          F.when(
              (F.col("nota_usuario") > 10) |
              (F.col("nota_usuario") < 0) | F.isnan("nota_usuario"),
              None
          )
          .otherwise(F.col("nota_usuario"))
      )
      #Validando a nota do usuario, considerando invalidos valores menores que 0 ou maiores que 10

      #A deduplicação ocorre nos valores originais, antes da normalização.

)

df_review.write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable("medallion.silver.tb_avaliacoes_usuarios") #Criação da tabela Silver sobrescrevendo os dados existentes

print("Tabela silver.tb_avaliacoes_usuarios criada com sucesso!") #Mensagem informando que a tabela Silver foi criada corretamente


In [0]:
#Essa seção do Notebook é focada na normalização da tabela bronze.tb_credits_and_tags

from pyspark.sql import functions as F

df_bronze = spark.table("medallion.bronze.tb_credits_and_tags") #Aqui estamos criando a variavel df_bronze que armazena a tabela bronze.tb_credits_and_tags

# Lista de gêneros aceitos
generos_validos = [
    "Action", "Adventure", "Animation", "Comedy", "Crime",
    "Documentary", "Drama", "Family", "Fantasy", "History",
    "Horror", "Music", "Mystery", "Romance", "Science Fiction",
    "TV Movie", "Thriller", "War", "Western"
] #Lista contendo os generos validos de acordo com os valores permitidos

df_generos = (
    df_bronze
    .select(
        F.col("id").cast("string").alias("id_filme"), #Converte a coluna id para string e renomeia para id_filme

        F.regexp_replace(
            F.col("genres"),
            r"[;|]",
            ","
        ).alias("generos") #Padroniza os separadores utilizados na coluna genres para virgula
    )
)

df_generos = df_generos.withColumn(
    "nome_genero",
    F.explode(F.split(F.col("generos"), ","))
) #Separa os diferentes generos armazenados na mesma celula, criando uma linha para cada genero

df_generos = (
    df_generos
    .withColumn(
        "nome_genero",
        F.trim(F.col("nome_genero"))
    ) #Remove espaços desnecessarios antes e depois do nome do genero

    .filter(
        F.col("nome_genero").isin(generos_validos)
    ) #Mantem somente os generos presentes na lista de valores validos
)

df_silver_generos = df_generos.dropDuplicates(
    ["id_filme", "nome_genero"]
) #Remove registros duplicados considerando o filme e o genero

df_silver_generos.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("medallion.silver.tb_generos") #Criação da tabela Silver sobrescrevendo os dados existentes

print("Tabela: silver.tb_generos criada com sucesso!") #Mensagem informando que a tabela Silver foi criada corretamente



In [0]:
#Essa seção do Notebook é focada na normalização de pessoas e empresas
from pyspark.sql import functions as F

df = spark.table("medallion.bronze.tb_credits_and_tags")

df_pessoas_empresas = (

    df.withColumnRenamed("id", "id_filme") #Traduzindo a coluna id para id_filme

    .withColumn(
        "nome_entidade",
        F.explode(
            F.split(
                F.col("cast"),
                r"[,;|]"
            )
        )
    ) #Separa os atores armazenados na coluna cast, criando uma linha para cada entidade

    .withColumn(
        "tipo_entidade",
        F.lit("Ator")
    ) #Define as entidades provenientes da coluna cast como Ator

    .select(
        "id_filme",
        "nome_entidade",
        "tipo_entidade"
    ) #Seleciona somente as colunas necessarias para a tabela Silver
)


#Criação da tabela temporaria com os atores


df_diretores = (

    df.withColumnRenamed("id", "id_filme") #Traduzindo a coluna id para id_filme

    .withColumn(
        "nome_entidade",
        F.explode(
            F.split(
                F.col("directors"),
                r"[,;|]"
            )
        )
    ) #Separa os diretores armazenados na coluna directors, criando uma linha para cada entidade

    .withColumn(
        "tipo_entidade",
        F.lit("Diretor")
    ) #Define as entidades provenientes da coluna directors como Diretor

    .select(
        "id_filme",
        "nome_entidade",
        "tipo_entidade"
    ) #Seleciona somente as colunas necessarias para a tabela Silver
)


#Criação da tabela temporaria com os diretores


df_roteiristas = (

    df.withColumnRenamed("id", "id_filme") #Traduzindo a coluna id para id_filme

    .withColumn(
        "nome_entidade",
        F.explode(
            F.split(
                F.col("writers"),
                r"[,;|]"
            )
        )
    ) #Separa os roteiristas armazenados na coluna writers, criando uma linha para cada entidade

    .withColumn(
        "tipo_entidade",
        F.lit("Roteirista")
    ) #Define as entidades provenientes da coluna writers como Roteirista

    .select(
        "id_filme",
        "nome_entidade",
        "tipo_entidade"
    ) #Seleciona somente as colunas necessarias para a tabela Silver
)


#Criação da tabela temporaria com os roteiristas


df_produtoras = (

    df.withColumnRenamed("id", "id_filme") #Traduzindo a coluna id para id_filme

    .withColumn(
        "nome_entidade",
        F.explode(
            F.split(
                F.col("production_companies"),
                r"[,;|]"
            )
        )
    ) #Separa as produtoras armazenadas na coluna production_companies, criando uma linha para cada entidade

    .withColumn(
        "tipo_entidade",
        F.lit("Produtora")
    ) #Define as entidades provenientes da coluna production_companies como Produtora

    .select(
        "id_filme",
        "nome_entidade",
        "tipo_entidade"
    ) #Seleciona somente as colunas necessarias para a tabela Silver
)


#Criação da tabela temporaria com as produtoras


df_pessoas_empresas = (

    df_pessoas_empresas

    .unionByName(df_diretores)

    .unionByName(df_roteiristas)

    .unionByName(df_produtoras)

) #Unificação das quatro fontes de entidades em uma única tabela





#Padroniza os nomes antes de filtrar, na mesma ordem da referência.
df_pessoas_empresas = (
    df_pessoas_empresas
    .withColumn("nome_entidade", F.initcap(F.trim(F.col("nome_entidade"))))
    .filter(
        F.col("nome_entidade").isNotNull()
        & (F.col("nome_entidade") != "")
        & (~F.col("nome_entidade").rlike(r"^\d+(\.\d+)?$"))
        & (~F.upper(F.col("nome_entidade")).isin(
            "[]", "N/A", "UNKNOWN", "NÃO INFORMADO", "NULL", "NONE"
        ))
        & (~F.lower(F.col("nome_entidade")).rlike(r"^/|.*\.(jpg|jpeg|png|webp)$"))
        & (F.length(F.col("nome_entidade")) <= 100)
    )
)

df_pessoas_empresas = (

    df_pessoas_empresas

    .dropDuplicates(
        [
            "id_filme",
            "nome_entidade",
            "tipo_entidade"
        ]
    )

) #Remove registros duplicados considerando o filme, a entidade e seu tipo


df_pessoas_empresas.write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable("medallion.silver.tb_pessoas_empresas") #Criação da tabela Silver sobrescrevendo os dados existentes

print("Tabela silver.tb_pessoas_empresas criada corretamente") #Mensagem informando que a tabela Silver foi criada corretamente


In [0]:
#Essa seção do Notebook é focada na normalização da tabela bronze.tb_cotacao_dolar
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
df = spark.table("medallion.bronze.tb_cotacao_dolar")

#Define exatamente as sete datas da execução: hoje e os seis dias anteriores.
hoje = datetime.now(ZoneInfo("America/Recife")).date()
inicio = (hoje - timedelta(days=7)).strftime("%Y-%m-%d")
fim = hoje.strftime("%Y-%m-%d")

#Converte o horário e consolida a cotação mais recente de cada dia.
df_cotacao = (
    df.withColumn("_hora_cotacao", F.expr("try_cast(dataHoraCotacao AS TIMESTAMP)"))
      .withColumn("data", F.to_date("_hora_cotacao"))
      .withColumn("cotacaoCompra", F.expr("try_cast(cotacaoCompra AS DECIMAL(18,8))"))
      .filter(F.col("data").isNotNull() & (F.col("cotacaoCompra") > 0))
)
janela_dia = Window.partitionBy("data").orderBy(
    F.col("_hora_cotacao").desc(),
    F.col("ingestion_datetime").desc(),
    F.col("cotacaoCompra").desc()
)
df_cotacao = (
    df_cotacao.withColumn("_rn", F.row_number().over(janela_dia))
    .filter(F.col("_rn") == 1)
    .select("data", "cotacaoCompra")
)

#Busca uma cotação anterior para preencher o primeiro dia sem publicação.
cotacao_anterior = (
    df_cotacao.filter(F.col("data") < F.to_date(F.lit(inicio)))
    .orderBy(F.col("data").desc()).limit(1)
)

#Cria somente as sete datas do intervalo da execução.
df_datas = spark.sql(f"""
    SELECT explode(sequence(
        to_date('{inicio}'), to_date('{fim}'), interval 1 day
    )) AS data
""")
df_historico = cotacao_anterior.unionByName(
    df_datas.join(df_cotacao, "data", "left").select("data", "cotacaoCompra")
)

#Forward fill para fins de semana e feriados.
janela_ffill = Window.orderBy("data").rowsBetween(
    Window.unboundedPreceding, Window.currentRow
)
df_historico = (
    df_historico.withColumn(
        "cotacaoCompra", F.last("cotacaoCompra", ignorenulls=True).over(janela_ffill)
    )
    .filter(F.col("data").between(F.to_date(F.lit(inicio)), F.to_date(F.lit(fim))))
    .withColumn("ingestion_datetime", F.current_timestamp())
    .select("cotacaoCompra", "data", "ingestion_datetime")
)

#Não publica uma série incompleta.
if df_historico.filter(F.col("cotacaoCompra").isNull()).limit(1).count():
    raise ValueError("Não há cotação disponível para preencher os sete dias solicitados.")

df_historico.write.mode("overwrite").format("delta").option(
    "overwriteSchema", "true"
).saveAsTable("medallion.silver.tb_cotacao_dolar")

print("Tabela silver.tb_cotacao_dolar criada corretamente com os últimos 7 dias")
display(df_historico.orderBy("data"))

